# Secure Defaults: the three patterns LSCBench rewards

The LSCBench paper (Oct 10, 2026) found that coding agents pass 73.6-91.3% of tasks
functionally but only 16.4-25.1% are *also* secure — and that the failure mode is
**omission**: the safe API was available, the agent just never reached for it.

This notebook rebuilds the leaderboard math and demos the three secure-default
patterns from the paper's case studies. Run it, then steal the patterns.


In [ ]:
# Leaderboard recompute — numbers from the LSCBench project page (pass@1)
lb = {
    "Claude Fable 5":   (91.1, 25.1),
    "GPT-5.6-Sol":      (89.1, 24.7),
    "Qwen3.8-Max":      (89.3, 22.0),
    "GLM-5.3":          (87.1, 21.3),
    "DeepSeek-V4-Pro":  (73.6, 18.0),
    "Kimi-K3":          (91.3, 16.4),
}
print(f"{'model':<16} {'works':>6} {'secure':>7} {'secure share of working':>24}")
for m, (w, s) in sorted(lb.items(), key=lambda kv: -kv[1][1]):
    print(f"{m:<16} {w:>5.1f}% {s:>6.1f}% {100*s/w:>22.1f}%")

# The Kimi-K3 paradox: best functional rate, worst secure rate
best_works = max(lb, key=lambda m: lb[m][0])
worst_secure = min(lb, key=lambda m: lb[m][1])
assert best_works == "Kimi-K3" and worst_secure == "Kimi-K3"
print(f"\nParadox confirmed: {best_works} tops 'works' ({lb[best_works][0]}%) "
      f"and trails 'secure' ({lb[worst_secure][1]}%). Green tests say nothing about safety.")

# Aggregate: 66% of 2,700 runs = works but still vulnerable
assert 0.66 * 2700 == 1782
print("66% of 2,700 runs -> 1,782 working-but-vulnerable outputs.")


## Pattern 1 — Safe deserialization (the SnakeYAML case)

The benchmark's Java case: agents that used the default `Yaml()` constructor honored
every `!!class` tag in untrusted YAML. The Python twin is `yaml.load` with an
unsafe loader vs `yaml.safe_load`. **Never execute the payload** — the safe loader
refuses to even construct it:


In [ ]:
import yaml

# A hostile config: asks the deserializer to instantiate an arbitrary class.
payload = "cmd: !!python/object/apply:os.system ['echo harmless-payload']"

try:
    yaml.safe_load(payload)
    raise AssertionError("safe_load should have refused this")
except yaml.constructor.ConstructorError as e:
    print("safe_load refused:", type(e).__name__)

# The unsafe loader would construct it (we do NOT call it — this is the point).
print("Loader=yaml.UnsafeLoader would instantiate the object. safe_load won't.")


## Pattern 2 — Argument lists, not shell strings (the ProcessBuilder case)

Four of six models passed mount args to `ProcessBuilder` as a list; two built a
string for `sh -c` — and the shell executed the user's "argument" as commands.
Same rule in every language: **never build a shell string from input.**


In [ ]:
import subprocess

def run_mount_list(mnt):          # the safe pattern: argv list, no shell
    return subprocess.run(["echo", "mounting", mnt],
                          capture_output=True, text=True).stdout

def run_mount_shell(mnt):         # the LSCBench failure: string -> shell
    return subprocess.run(f"echo mounting {mnt}", shell=True,
                          capture_output=True, text=True).stdout

evil = "/data; echo INJECTED"     # hostile "argument" (payload is a harmless echo)

out_list = run_mount_list(evil)
out_shell = run_mount_shell(evil)
assert "INJECTED" not in out_list.replace(evil, ""), "argv list must not execute input"
print("argv list :", out_list.strip())
print("sh -c str :", out_shell.strip().replace("\n", " | "))
assert "INJECTED" in out_shell, "shell form executes the injection (demo)"
print("\nList form treats hostile input as a literal string. Shell form runs it.")


## Pattern 3 — Confine paths and URIs (the SSRF / LocalAI case)

GPT-5.6-Sol's `GetURI` read any path it was handed, `file://` included. The reference
fix confines reads to the models directory. This is that fix, plus a regression
test — the exact kind of "replay the attack" test LSCBench runs:


In [ ]:
from pathlib import Path

MODELS = Path("models").resolve()

def resolve_model(uri: str) -> Path:
    if "://" in uri:
        raise ValueError("plain model names only")
    p = (MODELS / uri).resolve()
    if not p.is_relative_to(MODELS):
        raise ValueError("outside the models dir")
    return p

# the attack replay: must all raise
for hostile in ["file://models/llama.bin", "../secrets.txt",
                "..%2f..%2fetc/passwd".replace("%2f", "/")]:
    try:
        resolve_model(hostile)
        raise AssertionError(f"should have rejected {hostile!r}")
    except ValueError as e:
        print(f"rejected {hostile!r}: {e}")

# the friendly user still works
assert resolve_model("llama-3.bin") == MODELS / "llama-3.bin"
print("accepted 'llama-3.bin' ->", resolve_model("llama-3.bin"))


## The reminder effect

Appending "while also being secure" to the prompt nearly doubled secure rates for
three of four re-tested models — but left most tasks open (paper's verdict).


In [ ]:
reminder = {
    "GPT-5.6-Sol":     (24.7, 46.2),
    "GLM-5.3":         (21.3, 38.2),
    "Qwen3.8-Max":     (22.0, 37.6),
    "DeepSeek-V4-Pro": (18.0, 19.1),
}
for m, (b, a) in reminder.items():
    print(f"{m:<16} {b:>5.1f}% -> {a:>5.1f}%   (+{a-b:.1f}pp)")

gains = {m: a - b for m, (b, a) in reminder.items()}
assert gains["GPT-5.6-Sol"] > 20 and gains["DeepSeek-V4-Pro"] < 2
print("\nThree models gained +16..+22pp; DeepSeek-V4-Pro barely moved (+1.1pp).")
print("And even the best rerun (46.2%) leaves most of the 450 tasks insecure.")
